In [ ]:
from ac_segmentation.methods.skeletonize_array import break_branches, TS_skeletonize_volume
from ac_segmentation.utils.tensorstore import *
from ac_segmentation.utils.io import write_cv_skels_tar
from ac_segmentation.utils.h5_skeletons import *
from ac_segmentation.utils.h5_reconnect import *

from cloudvolume import CloudVolume, Skeleton, paths
import uuid

In [ ]:
###Create input tensor
arr = np.random.rand(100,100,100).astype('uint8')
input_tensor = create_tensor(fpath="input_vol/0", 
                             arr_shape=(100,100,100), 
                             dtype='uint8', 
                             chunk_shape=[64, 64, 64], 
                             driver='zarr3', 
                             codecs={"name": "blosc", "configuration": {"cname": "lz4", "clevel": 4}}, 
                             sharded=True, 
                             shard_factor=16)
input_tensor[...].write(arr).result()

In [ ]:
###Run skeletonization
skels = TS_skeletonize_volume(input_tensor, 
                              chunk_size=[100,100,100], 
                              n_jobs=10, prob_thresh=10, 
                              label_size_threshold=10, 
                              overlap=4, 
                              cutout=None)      
skels = [x for x in skels if x]
        
if len(skels) > 0 :       
    for ind in range(len(skels)):
        skels[ind].id = int(uuid.uuid4().int % 1e14)  
        print("Completed skeletonization, # of skels", len(skels))              
            
        #break branches
        skels = list(break_branches(skels).values())
                                                                                                                       
        #merge overlap skeletons
        fused = Skeleton.simple_merge(skels).consolidate().components()         
                    
        #remove twigs
        fused= prune_to_furthest_end_path(fused)      
        for ind in range(len(fused)):
            fused[ind].id = int(uuid.uuid4().int % 1e14)          
                                          
        #write raw swc       
        os.makedirs("skeleton_output", exist_ok=True)                                           
        write_cv_skels_tar('skeleton_output/skeletons_raw.swcs.tar.gz', fused, mode='w:gz')